In [ ]:
!pip install rouge_score

In [ ]:
import pandas as pd
import numpy as np
import ast
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from tqdm import tqdm


In [ ]:
train_file = "Dataset\complete_audio_features_train.parquet"
validation_file = "Dataset\complete_audio_features_validation.parquet"
test_file = "Dataset\complete_audio_features_test.parquet"
df_train = pd.read_parquet(train_file)
df_val = pd.read_parquet(validation_file)
df_test = pd.read_parquet(test_file)
df_all = pd.concat([df_train, df_test, df_val], ignore_index=True)

In [ ]:
def to_numpy_array(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x)
    if isinstance(x, str):
        return np.array(ast.literal_eval(x))
    raise ValueError(f"Unsupported type: {type(x)}")
df_all['embedding'] = df_all['embedding'].apply(to_numpy_array)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
MAX_CAPTION_LEN = 40
def shorten_caption(caption, tokenizer=tokenizer, max_len=MAX_CAPTION_LEN):
    tokens = tokenizer.tokenize(caption)
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    return tokenizer.convert_tokens_to_string(tokens)

df_all['caption'] = df_all['caption'].apply(lambda cap: shorten_caption(cap))


In [ ]:
classic_features = ['spectral_centroid', 'spectral_bandwidth', 'zero_crossing_rate', 'rms', 'tempo']
X_classic = df_all[classic_features].values
X_embed = np.vstack(df_all['embedding'].values)
X_combined = np.hstack([X_classic, X_embed])
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X_combined)
captions = df_all['caption'].values
names = df_all['name'].values

In [ ]:
X_temp, X_test, captions_temp, captions_test, names_temp, names_test = train_test_split(
    X_normalized, captions, names, test_size=0.15, random_state=42)
X_train, X_val, captions_train, captions_val, names_train, names_val = train_test_split(
    X_temp, captions_temp, names_temp, test_size=0.1765, random_state=42)

In [ ]:
MAX_LEN = MAX_CAPTION_LEN  # 40

In [ ]:
class AudioCaptionDataset(Dataset):
    def __init__(self, features, captions):
        self.features = features
        self.captions = captions

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        caption = self.captions[idx]
        tokens = tokenizer(caption, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)
        return x, input_ids, attention_mask

In [ ]:
train_loader = DataLoader(AudioCaptionDataset(X_train, captions_train), batch_size=64, shuffle=True)
val_loader = DataLoader(AudioCaptionDataset(X_val, captions_val), batch_size=64)
test_loader = DataLoader(AudioCaptionDataset(X_test, captions_test), batch_size=64)

In [ ]:
class CaptionGenerator(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, lstm_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=lstm_layers, batch_first=True, bidirectional=False)
        self.decoder = T5ForConditionalGeneration.from_pretrained("t5-small")

    def forward(self, audio_feat, input_ids=None, attention_mask=None, labels=None):
        lstm_out, _ = self.lstm(audio_feat.unsqueeze(1))  # (batch, seq_len=1, hidden)
        encoder_hidden_state = lstm_out
        output = self.decoder(
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_state),
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CaptionGenerator(input_dim=X_normalized.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
EPOCHS = 20
best_val_bleu = 0
patience = 5
patience_counter = 0

In [ ]:
def evaluate_model(loader):
    model.eval()
    preds, refs = [], []
    for audio, input_ids, attn_mask in loader:
        audio = audio.to(device)
        with torch.no_grad():
            lstm_out, _ = model.lstm(audio.unsqueeze(1))
            encoder_output = BaseModelOutput(last_hidden_state=lstm_out)
            generated_ids = model.decoder.generate(encoder_outputs=encoder_output, max_length=MAX_LEN)
        pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        true_texts = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        preds.extend(pred_texts)
        refs.extend(true_texts)

    smoothie = SmoothingFunction().method4
    refs_for_bleu = [[ref.split()] for ref in refs]
    preds_for_bleu = [pred.split() for pred in preds]
    bleu_score = corpus_bleu(refs_for_bleu, preds_for_bleu, smoothing_function=smoothie)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    for pred, ref in zip(preds, refs):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    print(f"\nBLEU: {bleu_score:.4f} | ROUGE-1: {np.mean(rouge1_scores):.4f} | ROUGE-2: {np.mean(rouge2_scores):.4f} | ROUGE-L: {np.mean(rougeL_scores):.4f}")
    return bleu_score

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for audio, input_ids, attn_mask in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        audio, input_ids, attn_mask = audio.to(device), input_ids.to(device), attn_mask.to(device)
        labels = input_ids.clone()
        optimizer.zero_grad()
        outputs = model(audio, input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

    val_bleu = evaluate_model(val_loader)
    if val_bleu > best_val_bleu:
        best_val_bleu = val_bleu
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        print("Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Final Test Evaluation
print("\nFinal Test Evaluation:")
model.load_state_dict(torch.load("best_model.pt"))
evaluate_model(test_loader)

In [ ]:
def generate_caption(audio_feature):
    model.eval()
    audio_tensor = torch.tensor(audio_feature, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        lstm_out, _ = model.lstm(audio_tensor.unsqueeze(1))
        encoder_output = BaseModelOutput(last_hidden_state=lstm_out)
        generated = model.decoder.generate(encoder_outputs=encoder_output, max_length=MAX_LEN)
        caption = tokenizer.decode(generated[0], skip_special_tokens=True)
    return caption


In [ ]:
sample_idx = 20
predicted_caption = generate_caption(X_test[sample_idx])
print("\nSample Prediction")
print("Music File:", names_test[sample_idx])
print("Predicted Caption:", predicted_caption)
print("Real Caption:", captions_test[sample_idx])